Note: The function below is designed to run in the D\_alt\_Data\_county\_typical\_daily\_PM25\_analysis notebook after establishing the contiguous counties. It successfully grabs a single median cell per day but has not accounted for the fact that many median cells will tie for median. The script only grabs the first cell, biasing the median position to the north of each county. Likely, I will want to grab each median cell to establish the typical median cell area.


#### Identify the Daily Median Alternative PM2.5 Value Location in Each Contiguous U.S. County



In [0]:
#Purpose of Cell Block:
#SELF-CONTAINED county-scale diagnostic for get_median_cell_xy(), extended to:
# - Zoom to county extent 
# - Extract all raster cells intersecting county[i]
# - Identify the selected median cell (production logic)
# - Recolor extracted cells by WITHIN-COUNTY PM2.5 rank (greatest → least)
#   to visualize likely median candidates and check for boundary artifacts

library(terra)
library(sf)
library(exactextractr)
library(dplyr)
library(tibble)


AltPM25_file <- altPM25_files[1]  # choosing one raster day explicitly
i <- 1                            # choosing one county index explicitly

cat("Diagnostic raster:", basename(AltPM25_file), "\n")
cat("Diagnostic county row:", i, "\n")

#--------------------------------------------------
# 1. Read raster and counties (exactly as production)
#--------------------------------------------------
rastAltPM25_file <- rast(AltPM25_file)

counties_sf0 <- if (inherits(contiguous_counties, "sf")) {
  contiguous_counties
} else {
  st_as_sf(contiguous_counties)
}

counties_sf <- st_transform(counties_sf0, crs(rastAltPM25_file))

#--------------------------------------------------
# 2. Compute county medians for this day (production logic)
#--------------------------------------------------
county_median <- exact_extract(
  rastAltPM25_file,
  counties_sf,
  "median"
)

county_i <- counties_sf[i, ]
median_value <- county_median[i]

#--------------------------------------------------
# 3. COUNTY-SCALE ZOOM (critical for visibility)
#--------------------------------------------------
county_vect <- vect(county_i)
county_ext  <- ext(county_vect)

pad_x <- 0.1 * (county_ext[2] - county_ext[1])
pad_y <- 0.1 * (county_ext[4] - county_ext[3])

zoom_ext <- ext(
  county_ext[1] - pad_x,
  county_ext[2] + pad_x,
  county_ext[3] - pad_y,
  county_ext[4] + pad_y
)

rast_zoom <- crop(rastAltPM25_file, zoom_ext)

#--------------------------------------------------
# 4. Baseline: zoomed raster + county boundary
#--------------------------------------------------
plot(
  rast_zoom,
  main = paste("Zoomed raster + county boundary |",
               basename(AltPM25_file),
               "| county row", i)
)
plot(st_geometry(county_i), add = TRUE, border = "black", lwd = 2)

#--------------------------------------------------
# 5. Extract all raster cells intersecting county[i]
#--------------------------------------------------
cells_i <- exact_extract(
  rastAltPM25_file,
  county_i,
  include_cell = TRUE
)[[1]]

cells_i_pts <- cbind(
  cells_i,
  xyFromCell(rastAltPM25_file, cells_i$cell)
) |>
  as_tibble()   # xyFromCell already returns columns named x, y

#--------------------------------------------------
# 6. Identify the median cell (production logic)
#--------------------------------------------------
cells_i_diff <- cells_i_pts |>
  filter(!is.na(value)) |>
  mutate(diff = abs(value - median_value))

median_cell <- cells_i_diff |>
  arrange(diff) |>
  slice(1)

#--------------------------------------------------
# 7. QA: verify xyFromCell() consistency (name-safe)
#--------------------------------------------------
median_xy <- xyFromCell(
  rastAltPM25_file,
  median_cell$cell
)

stopifnot(
  unname(median_xy[1, 1]) == median_cell$x,
  unname(median_xy[1, 2]) == median_cell$y
)

#--------------------------------------------------
# 8. RECOMMENDED DIAGNOSTIC:
#    Recolor extracted cells by WITHIN-COUNTY PM2.5 rank
#    (highest values = red, lowest = blue)
#--------------------------------------------------
cells_i_ranked <- cells_i_pts |>
  filter(!is.na(value)) |>
  mutate(
    rank_desc = rank(-value, ties.method = "first"),
    rank_pct  = rank_desc / max(rank_desc)
  )

pal <- colorRampPalette(c("red", "orange", "yellow", "green", "blue"))
col_vec <- pal(100)[ceiling(cells_i_ranked$rank_pct * 100)]

plot(
  rast_zoom,
  main = "Within-county raster cells ranked by PM2.5 value"
)
plot(st_geometry(county_i), add = TRUE, border = "black", lwd = 2)

points(
  cells_i_ranked$x,
  cells_i_ranked$y,
  pch = 20,
  cex = 0.7,
  col = col_vec
)

# highlight the selected median cell
points(
  median_cell$x,
  median_cell$y,
  pch = 21,
  bg = "black",
  cex = 1.9
)

#--------------------------------------------------
# Interpretation notes:
# - Red cells = highest PM2.5 within county
# - Blue cells = lowest
# - Green/yellow band ≈ median candidates
# - Black dot = cell selected by get_median_cell_xy()
# If the black dot hugs the boundary while similarly ranked cells
# exist in the interior, investigate geometry/weighting artifacts.
#--------------------------------------------------


In [0]:
#Purpose of Cell Block:
#Numerical + spatial audit of county-level median selection.
#Produces a table of ALL raster cells intersecting county[i],
#computes the TRUE median directly, and verifies the selected median cell.

library(terra)
library(sf)
library(exactextractr)
library(dplyr)
library(tibble)


AltPM25_file <- altPM25_files[1]   # one raster day
i <- 1                             # one county index

#--------------------------------------------------
# 1. Read raster and counties (production-equivalent)
#--------------------------------------------------
rastAltPM25_file <- rast(AltPM25_file)

counties_sf0 <- if (inherits(contiguous_counties, "sf")) {
  contiguous_counties
} else {
  st_as_sf(contiguous_counties)
}

counties_sf <- st_transform(counties_sf0, crs(rastAltPM25_file))
county_i <- counties_sf[i, ]

#--------------------------------------------------
# 2. Extract ALL raster cells intersecting county[i]
#--------------------------------------------------
cells_i <- exact_extract(
  rastAltPM25_file,
  county_i,
  include_cell = TRUE
)[[1]]

cells_tbl <- cbind(
  cells_i,
  xyFromCell(rastAltPM25_file, cells_i$cell)
) |>
  as_tibble() |>
  filter(!is.na(value))

#--------------------------------------------------
# 3. Compute the TRUE county median from extracted values
#--------------------------------------------------
true_median_value <- median(cells_tbl$value)

#--------------------------------------------------
# 4. Rank cells and compute distance from true median
#--------------------------------------------------
cells_tbl <- cells_tbl |>
  mutate(
    abs_diff_from_true_median = abs(value - true_median_value)
  ) |>
  arrange(abs_diff_from_true_median)

#--------------------------------------------------
# 5. Identify:
#    - exact median matches (if any)
#    - closest median candidates
#--------------------------------------------------
exact_median_cells <- cells_tbl |>
  filter(value == true_median_value)

closest_median_cells <- cells_tbl |>
  slice_min(abs_diff_from_true_median, n = 5)

#--------------------------------------------------
# 6. Reproduce algorithm’s selected median cell
#--------------------------------------------------
county_median_vector <- exact_extract(
  rastAltPM25_file,
  counties_sf,
  "median"
)

algo_selected <- cells_tbl |>
  mutate(diff_algo = abs(value - county_median_vector[i])) |>
  arrange(diff_algo) |>
  slice(1)

#--------------------------------------------------
# 7. AUDIT TABLE
#--------------------------------------------------
audit_table <- bind_rows(
  algo_selected |> mutate(selection = "Algorithm-selected"),
  closest_median_cells |> mutate(selection = "Closest-to-true-median")
) |>
  select(
    selection,
    cell,
    value,
    abs_diff_from_true_median,
    x,
    y
  )

#--------------------------------------------------
# 8. Print results for inspection
#--------------------------------------------------
cat("\nTRUE county median value:\n")
print(true_median_value)

cat("\nAlgorithm-selected median cell:\n")
print(algo_selected |> 
        select(cell, value, abs_diff_from_true_median, x, y))

cat("\nTop closest cells to true median:\n")
print(closest_median_cells |>
        select(cell, value, abs_diff_from_true_median, x, y))

cat("\nAudit comparison table:\n")
print(audit_table)


In [0]:
#Purpose: Description  Identify, for each county and each day, the raster cell whose PM2.5 value is closest to the county-level median. For every county–day combination, records the cell index (for identication), the median-cell PM2.5 value, the absolute deviation from the county median (for QA purposes), and the cell’s x/y coordinates (for my later assessment). Due to memory limitations, the workflow is designed to be restartable (i.e., my results are written to date-stamped checkpoint files so previously completed days are skipped).

#BEGIN FUNCTION: GET MEDIAN CELL FOR EACH COUNTY ON GIVEN DATE   
get_median_cell_xy <- function(i, rastAltPM25_file, counties_sf, county_median) {
    #extract a dataframe of cells within county[i], including the cells value and index
    cells_i <- exact_extract(
            rastAltPM25_file,
            counties_sf[i, ],
            include_cell = TRUE
        )[[1]]

    median_value <- county_median[i] #identify the median value for county[i]

    median_county_cell <- cells_i %>%
        filter(!is.na(value)) %>% #remove all PM2.5 values that are NA
        mutate(diff = abs(value - median_value)) %>% #Add new column called "diff" that calculates the absolute value of PM2.5 versus our median above
        arrange(diff) %>% #organize diff column so value closest to median appears at the top
        slice(1) %>% #grab only the value closet to median
        select(cell, value, diff) #extract only relevant cell

    median_county_xy <- xyFromCell(rastAltPM25_file, median_county_cell$cell) #from the median cell we identified, grab the xy

    tibble( #create a tibble that preserves the county row[i], the median cell index, and the xy coords for the median cell
        county_row = i,
        cell = median_county_cell$cell,
        median_cell_value = median_county_cell$value,
        abs_diff_from_county_median = median_county_cell$diff,
        x = median_county_xy[1, 1],
        y = median_county_xy[1, 2]
  )
}
#END FUNCTION: GET MEDIAN CELL FOR EACH COUNTY ON GIVEN DATE

#BEGIN FUNCTION: EXTRACT DATE
get_date_from_filename <- function(filepath) {
    str_extract(basename(filepath), "[0-9]{8}")
}
#END FUNCTION: EXTRACT DATE

#BEGIN FUNCTION: CHECKPOINT FILENAME AND PATH
checkpoint_path <- function(AltPM25_file) {
    date_str <- get_date_from_filename(AltPM25_file)
    file.path(checkpoint_dir, paste0("median_cells_", date_str, ".csv"))
}
#END FUNCTION: CHECKPOINT FILENAME AND PATH

counties_sf0 <- if (inherits(contiguous_counties, "sf")) contiguous_counties else st_as_sf(contiguous_counties)

checkpoint_dir <- "D_processed_data/intermediate_scratch/checkpoints"

existing_checkpoints <- list.files( #feed the checkpoint filepaths into a string to check the last successfull export
  checkpoint_dir,
  pattern = "^median_cells_[0-9]{8}\\.csv$",
  full.names = FALSE
)

completed_dates <- str_extract(existing_checkpoints, "[0-9]{8}") #extract dates from the checkpoint filepaths

all_dates <- get_date_from_filename(altPM25_files) #run get date function on all PM2.5 files

altPM25_files_todo <- altPM25_files[!(all_dates %in% completed_dates)] #extract all files that are not in the checkpoint filepaths

#check the total number of files versus the files i still need to work on
length(altPM25_files)
length(altPM25_files_todo) 

#check the first file that will be work on
head(get_date_from_filename(altPM25_files_todo), n=1)

n_days <- length(altPM25_files) #used for assessing progress

day_counter <- length(altPM25_files) - length(altPM25_files_todo)

for (AltPM25_file in altPM25_files_todo) { #iterate each AltPM25_file (i.e., day iterator)...
    
    day_counter <- day_counter + 1

    cat(
        sprintf(
            "\nProcessing day %d of %d: %s\n",
            day_counter,
            n_days,
            basename(AltPM25_file)
            )
        )

    flush.console()    
    
    rastAltPM25_file <- rast(AltPM25_file) #read in the raster from the day dataset above
    
    counties_sf <- st_transform(counties_sf0, crs(rastAltPM25_file)) #using our read in raster, reproject counties to align coordinate reference systems
    
    county_median <- exact_extract(rastAltPM25_file, counties_sf, "median") #use zonal stats to extract the median PM2.5 value for each county on day[AltPM25_file]
    
    daily_results <- tibble() #create container to hold one day of results
    
    for (i in seq_len(nrow(counties_sf))) { #iterate each county...
        one_row <- get_median_cell_xy(i, rastAltPM25_file, counties_sf, county_median) #run function that grabs median cell for each county 
        
        daily_results <- bind_rows(daily_results, one_row) #bind each county results into a dataset for the day
    }      
        
    daily_results <- daily_results %>%
        mutate(file = AltPM25_file) #create a new column that tracks the filepath used

    checkpoint_csv <- checkpoint_path(AltPM25_file) #run function to translate the current raster filepath to checkpoint filepath
    
    tmp_csv <- paste0(checkpoint_csv, ".tmp") #concat a .tmp extension, necessary to minimize risk the program partially exported a file when cocalc shutdown
    
    write_csv(daily_results, tmp_csv) #write the temp file to checkpoint folder
    
    file.rename(tmp_csv, checkpoint_csv) #rename the temp file as the final filename
}

In [0]:
#Purpose of Cell Block: Grab all per-day checkpoint CSV files produced by the median-cell extraction workflow above, read each file into memory, and vertically concatenate them into a single multi-day dataset. Then validate number of unique day files represented in the combined results, ensuring that all expected daily outputs were successfully merged.

getwd()
setwd("/home/user/capstone/A_data")
checkpoint_files <- list.files( #create vector of checkpoint files
    "D_processed_data/intermediate_scratch/checkpoints", pattern = "\\.csv$", full.names = TRUE) 

print("Number of days in checkpoint folder")
n_distinct(checkpoint_files) #count alt. PM2.5 files
  
all_days_list <- list() #initialize list for storing PM2.5 files

for (i in seq_along(checkpoint_files)) { #for each checkpoint file in the vector above...
    all_days_list[[i]] <- read_csv( #read in csv[i] as a tibble (see next row)
        checkpoint_files[i], 
        show_col_types = FALSE 
    )
}

results_all_days <- bind_rows(all_days_list) #bind tibbles created in the for loop above

head(results_all_days) #tentatively confirm bind worked
file_check <- n_distinct(results_all_days$file)
print("Number of days in bound dataset")
file_check

#Conclusion: Dataset was successfully generated